### Imports

In [1]:
import os
from pathlib import Path
import json
import cv2
import supervision as sv
from tqdm import tqdm
import sys
import numpy as np

### Initialization

In [2]:
sys.path.insert(0, "../../")
from config import DATASETS_PATH, IMG_SHAPE, SEED

IMG_TARGET_SIDE = IMG_SHAPE[0]
ANNOTATED_DATASET_PATH = os.path.join(DATASETS_PATH, 'cropped', 'annotated')

### Functions

In [3]:
def crop_from_center_with_padding(image, x1, y1, x2, y2, crop_width=300, crop_height=300, draw_bbox=False):
    """
    Crops a fixed-size area, adding padding if the crop exceeds image boundaries.
    Guarantees the output image size is exactly crop_width x crop_height.
    Optionally draws a green rectangle on the original bounding box area.

    :param image: Input image (NumPy array).
    :param x1, y1, x2, y2: Coordinates of the original bounding box.
    :param crop_width: Desired width of the output crop.
    :param crop_height: Desired height of the output crop.
    :param draw_bbox: If True, draws a green rectangle on the original bounding box.
    :return: Cropped image (300x300) and optionally the original image with the drawn bbox.
    """
    
    # 1. Find the center of the original bounding box
    center_x = (x1 + x2) / 2
    center_y = (y1 + y2) / 2
    
    # 2. Define the new 300x300 box coordinates
    half_width = crop_width / 2
    half_height = crop_height / 2
    
    new_x1 = center_x - half_width
    new_y1 = center_y - half_height
    new_x2 = center_x + half_width
    new_y2 = center_y + half_height
    
    # 3. Calculate necessary padding
    img_height, img_width, _ = image.shape
    
    pad_left = int(max(0, -new_x1))
    pad_top = int(max(0, -new_y1))
    pad_right = int(max(0, new_x2 - img_width))
    pad_bottom = int(max(0, new_y2 - img_height))
    
    # 4. Add padding to the image.    
    padded_image = cv2.copyMakeBorder(
        image, pad_top, pad_bottom, pad_left, pad_right, 
        cv2.BORDER_CONSTANT, value=[0, 0, 0] # Black padding
    )
    
    # 5. Adjust crop coordinates for the new padded image
    final_x1 = int(new_x1 + pad_left)
    final_y1 = int(new_y1 + pad_top)
    final_x2 = int(new_x2 + pad_left)
    final_y2 = int(new_y2 + pad_top)
    
    # 7. Add optional bounding box drawing
    if draw_bbox:
        # Define the color green in BGR format (OpenCV's default)
        green_color = (0, 255, 0) # (Blue, Green, Red)
        thickness = 3             # Thickness of the rectangle border
        
        # Draw the rectangle on the copied image
        cv2.rectangle(padded_image, (x1, y1), (x2, y2), green_color, thickness)

    # 6. Perform the crop on the padded image
    cropped_image = padded_image[final_y1:final_y2, final_x1:final_x2]

    return cropped_image


In [4]:
def crop_and_save_detections(images_dir: Path, annotations_path: Path, output_dir: Path, resize_factor, area_data, crop_from_center= False) -> None:
    """
    Loads COCO annotations, crops the detected objects from images, and saves
    them into class-specific folders.

    Args:
        images_dir (Path): The path to the directory containing the images.
        annotations_path (Path): The path to the COCO JSON annotation file.
        output_dir (Path): The path to the directory where cropped images will be saved.
    """
    # --- Validations ---
    if not images_dir.is_dir():
        print(f"Images dir not found: {images_dir}")
        return
    if not annotations_path.is_file():
        print(f"Annotations file not found: {annotations_path}")
        return
    
    # Ensure the main output directory exists
    output_dir.mkdir(parents=True, exist_ok=True)

    # Load the dataset using supervision
    print("Loading dataset...")
    dataset = sv.DetectionDataset.from_coco(
        images_directory_path=str(images_dir),
        annotations_path=str(annotations_path),
    )

    print(f"Found {len(dataset.classes)} classes: {dataset.classes}")

    # Iterate through the dataset with a progress bar
    for image_path, image, detections in tqdm(dataset):
        if image is None:
            continue
        image_name = Path(image_path).stem
        image_group = image_name[0]
        # image_side = area_data[image_group]['lado_cuadrado']
        # image_resize_factor = int(resize_factor * image_side)

        # Iterate through each detection in the image
        for i, detection in enumerate(detections):
            # The detection object contains xyxy, mask, confidence, class_id, etc.
            xyxy, _, _, class_id, _, _ = detection

            # Get the class name for the current detection
            class_name = dataset.classes[class_id].lower()

            # Create a directory for the class if it doesn't exist
            class_dir = output_dir / class_name 
            class_dir.mkdir(parents=True, exist_ok=True)

            # Crop the detection from the image using its bounding box
            x1, y1, x2, y2 = map(int, xyxy)
            cropped_image = image[y1:y2, x1:x2] if not crop_from_center else crop_from_center_with_padding(image, x1, y1, x2, y2, 300, 300, True)

            # Ensure the cropped image is not empty before saving
            if cropped_image.size == 0:
                print(f"  - Skipping empty crop for detection {i} in {image_name}.png")
                continue

            cropped_image = cv2.resize(cropped_image, (IMG_TARGET_SIDE, IMG_TARGET_SIDE)) if not crop_from_center else cropped_image
            # x, y, w, h = segmentators.CellMaskGenerator.adjust_bbox(segmentators.CellMaskGenerator, x1, y1, x2-x1, y2-y1, image_resize_factor*image_resize_factor, image.shape[1], image.shape[0])
            # cropped_image = cv2.resize(image[y:y+h, x:x+w], (IMG_TARGET_SIDE, IMG_TARGET_SIDE))


            # Generate a unique filename for the cropped image
            cropped_image_filename = f"{image_name}_{i}.png"
            cropped_image_path = class_dir / cropped_image_filename

            # Save the cropped image
            cv2.imwrite(str(cropped_image_path), cropped_image)

    print(f"\n✅ Processing complete. Cropped images are saved in '{output_dir}'.")


In [5]:
def crop_and_save_detections_direct(images_dir: Path, annotations_path: Path, output_dir: Path) -> None:
    """
    Loads COCO annotations, crops the detected objects from images, and saves
    them using the annotation ID in the filename.

    Args:
        images_dir (Path): The path to the directory containing the images.
        annotations_path (Path): The path to the COCO JSON annotation file.
        output_dir (Path): The path to the directory where cropped images will be saved.
    """
    # --- Validations ---
    if not images_dir.is_dir():
        print(f"Images dir not found: {images_dir}")
        return
    if not annotations_path.is_file():
        print(f"Annotations file not found: {annotations_path}")
        return

    output_dir.mkdir(parents=True, exist_ok=True)

    print("Loading COCO annotations...")
    with open(annotations_path) as f:
        coco_data = json.load(f)

    # --- Create helper maps for quick lookups ---
    # 1. Map category ID to category name
    category_map = {cat['id']: cat['name'] for cat in coco_data.get('categories', [])}
    
    # 2. Map image ID to its file path and name
    image_map = {
        img['id']: {
            'path': images_dir / img['file_name'],
            'file_name': img['file_name']
        } 
        for img in coco_data['images']
    }

    print(f"Found {len(coco_data['annotations'])} annotations to process.")

    # --- Main loop: Iterate directly through each annotation ---
    for ann in tqdm(coco_data['annotations']):
        annotation_id = ann['id']
        image_id = ann['image_id']
        category_id = ann['category_id']
        bbox = ann['bbox'] # COCO format is [x_min, y_min, width, height]

        # Get image and class info from our maps
        image_info = image_map.get(image_id)
        class_name = category_map.get(category_id, 'unknown_class').lower()
        
        if not image_info:
            print(f"Warning: Skipping annotation {annotation_id} due to missing image_id {image_id}")
            continue

        # Create a directory for the class
        class_dir = output_dir / class_name
        class_dir.mkdir(parents=True, exist_ok=True)
        
        # Load the source image
        image_path = image_info['path']
        if not image_path.exists():
            continue
            
        image = cv2.imread(str(image_path))
        if image is None:
            continue

        # Crop the bounding box
        x, y, w, h = map(int, bbox)
        cropped_image = image[y : y+h, x : x+w]

        # Ensure the crop is valid before saving
        if cropped_image.size == 0:
            continue

        # (Optional) You can add your resize logic here if needed
        # IMG_TARGET_SIDE = 224 # Example size
        cropped_image = cv2.resize(cropped_image, (IMG_TARGET_SIDE, IMG_TARGET_SIDE))

        # Save the cropped image with the annotation ID in the name
        image_name_stem = Path(image_info['file_name']).stem
        cropped_image_filename = f"{image_name_stem}_{annotation_id}.png"
        cropped_image_path = class_dir / cropped_image_filename

        cv2.imwrite(str(cropped_image_path), cropped_image)

    print(f"\n✅ Processing complete. Cropped images are saved in '{output_dir}'.")

In [6]:
def remove_images_with_black_markings(directory_path: Path, black_threshold=10, white_threshold=245, percentage_threshold=1.0):
    """
    Identifies images in a directory that have a significant number of black pixels.

    Args:
        directory_path (Path): The path to the directory containing images.
        black_threshold (int): Pixel intensity value (0-255). Pixels below this
                               value are considered 'black'. Defaults to 10.
        percentage_threshold (float): The percentage of black pixels required
                                      to classify an image as having markings.
                                      (e.g., 1.0 for 1%). Defaults to 1.0.

    Returns:
        list: A list of filenames for images that have significant black markings.
    """
    images_with_markings = 0
    
    # A list of common image file extensions to check
    valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

    print(f"Scanning directory: {directory_path}")

    images_paths = [str(path) for path in directory_path.rglob('*')]    # Iterate over every file in the directory
    for filename in tqdm(images_paths):
        # Check for a valid image extension
        if not any(filename.lower().endswith(ext) for ext in valid_extensions):
            continue

        file_path = os.path.join(directory_path, filename)

        try:
            # Read the image using OpenCV
            image = cv2.imread(file_path)

            if image is None:
                print(f"Warning: Could not read image {filename}. Skipping.")
                continue

            # Convert the image to grayscale for easier processing
            gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

            # Calculate the total number of pixels
            total_pixels = gray_image.size

            # Count the number of pixels that are 'black' (below the threshold)
            black_pixels = np.sum(gray_image <= black_threshold)

            white_pixels = np.sum(gray_image >= white_threshold)

            # Calculate the percentage of black pixels
            percentage_of_solid_pixels = ((black_pixels + white_pixels) / total_pixels) * 100

            # If the percentage exceeds the threshold, remove image
            if percentage_of_solid_pixels > percentage_threshold:
                images_with_markings += 1
                os.remove(file_path)

        except Exception as e:
            print(f"Error processing {filename}: {e}")

    print(f"Removed {images_with_markings} images with markings")



In [7]:
def remove_repeated_images(dir):
    """
    Remove repeatead images in the new datasets from roboflow since between datasets images are repeatead but with slightly different names    
    """
    def get_base_name(filename):
        for ext in ["_jpg", "_png"]:
            idx = filename.find(ext)
            if idx != -1:
                return filename[:idx+4]  # include the extension marker
        return filename

    files_by_base = defaultdict(list)
    for dirpath, dirnames, filenames in os.walk(dir):
        for filename in filenames:
            if filename.lower().endswith('.json'):
                continue  # Skip .json files
            base = get_base_name(filename)
            files_by_base[base].append(os.path.join(dirpath, filename))

    for file_list in files_by_base.values():
        for file_to_delete in file_list[1:]:
            try:
                os.remove(file_to_delete)
                print(f"Deleted: {file_to_delete}")
            except Exception as e:
                print(f"Error deleting {file_to_delete}: {e}")

In [8]:
def extract_value(annotation_str):
    """
    Define a function to extract the value from the annotations column
    """
    annotation_list = json.loads(annotation_str.lower())
    return annotation_list[0]['value'] if annotation_list and 'value' in annotation_list[0] else None

def extract_filename(subject_data_str):
    """
    Parses a JSON string from the 'subject_data' column
    and extracts the 'Filename' value from the nested dictionary.
    """
    if not isinstance(subject_data_str, str):
        return None
    try:
        # Load the string as a JSON object
        data = json.loads(subject_data_str)
        if not data:
            return None # Handles empty JSON object '{}'
            
        # The outer dictionary has a dynamic key. We get its value,
        # which is the inner dictionary.
        inner_dict = next(iter(data.values()))
        
        # Return the value of the 'Filename' key, or None if it doesn't exist
        return inner_dict.get('Filename')
    except (json.JSONDecodeError, StopIteration, AttributeError):
        # Handle cases where the string is not valid JSON,
        # or doesn't have the expected structure.
        return None

### Generate classifications from all datasets

Warning: When croping images some images won't be found since they are not tagged and the supervision library will warn about making the output dirty but it will be cropping images that are found anyway

#### Crops images from ina coco annotations

In [12]:
IMAGES_DIR = Path(os.path.join(DATASETS_PATH, 'full_fov', 'processed', 'ina', 'images'))
COCO_ANNOTATIONS_PATH = Path(os.path.join(DATASETS_PATH, 'full_fov', 'processed', 'ina', 'annotations_coco.json'))

OUTPUT_DIR = Path(os.path.join(DATASETS_PATH, 'cropped', 'annotated', 'ina'))

crop_and_save_detections_direct(IMAGES_DIR, COCO_ANNOTATIONS_PATH, OUTPUT_DIR)

Loading COCO annotations...
Found 2791 annotations to process.


100%|██████████| 2791/2791 [01:54<00:00, 24.33it/s]


✅ Processing complete. Cropped images are saved in '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/cropped/annotated/ina'.


#### Crops images from onion_cell_merged coco annotations

In [ ]:
IMAGES_DIR = Path(os.path.join(DATASETS_PATH, 'full_fov', 'processed', 'onion_cell_merged', 'images'))
COCO_ANNOTATIONS_PATH = Path(os.path.join(DATASETS_PATH, 'full_fov', 'processed', 'onion_cell_merged', 'annotations_coco.json'))

OUTPUT_DIR = Path(os.path.join(DATASETS_PATH, 'cropped', 'annotated', 'onion_cell_merged'))

BLACK_PIXEL_INTENSITY_THRESHOLD = 10  # How dark a pixel must be to be 'black' (0-255)
WHITE_PIXEL_INTENSITY_THRESHOLD = 245 # How brigth a pixel must be to be 'white' (0-255)
BLACK_PIXEL_PERCENTAGE = 15.0    # What percentage of the image needs to be black or white


crop_and_save_detections_direct(IMAGES_DIR, COCO_ANNOTATIONS_PATH, OUTPUT_DIR)

# Remove images with black or white markings
marked_images = remove_images_with_black_markings(
    Path(OUTPUT_DIR),
    black_threshold=BLACK_PIXEL_INTENSITY_THRESHOLD,
    white_threshold=WHITE_PIXEL_INTENSITY_THRESHOLD,
    percentage_threshold=BLACK_PIXEL_PERCENTAGE
)

Loading COCO annotations...
Found 20518 annotations to process.


100%|██████████| 20518/20518 [01:51<00:00, 184.05it/s]



✅ Processing complete. Cropped images are saved in '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/cropped/annotated/onion_cell_merged'.
Scanning directory: /home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/cropped/annotated/onion_cell_merged


100%|██████████| 20513/20513 [00:10<00:00, 1897.03it/s]

Removed 957 images with markings


#### Crops images from new roboflow datasets coco annotations

In [14]:
IMAGES_DIR = Path(os.path.join(DATASETS_PATH, 'full_fov', 'processed', 'roboflow_datasets', 'images'))
COCO_ANNOTATIONS_PATH = Path(os.path.join(DATASETS_PATH, 'full_fov', 'processed', 'roboflow_datasets', 'annotations_coco.json'))

OUTPUT_DIR = Path(os.path.join(DATASETS_PATH, 'cropped', 'annotated', 'roboflow_datasets'))

crop_and_save_detections_direct(IMAGES_DIR, COCO_ANNOTATIONS_PATH, OUTPUT_DIR)

Loading COCO annotations...
Found 5523 annotations to process.


100%|██████████| 5523/5523 [00:01<00:00, 3846.49it/s]


✅ Processing complete. Cropped images are saved in '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/datasets/cropped/annotated/roboflow_datasets'.


#### Separate images tagged from zooniverse (currently unused)

In [ ]:
import glob
import pandas as pd
import shutil
import os
from tqdm import tqdm

# Configuration
CSV_FILE_PATH = os.path.join(DATASETS_PATH, 'cropped', 'zooniverse', 'automating-allium-cepa-assay-analysis-with-ai-classifications.csv')
INA_CROPS_PATH = os.path.join(DATASETS_PATH, 'cropped', 'ina', 'images')
RB_CROPS_PATH = os.path.join(DATASETS_PATH, 'onion_cell_merged', 'images')
INA_CROPS = glob.glob(os.path.join(INA_CROPS_PATH, '*'))
RB_CROPS = glob.glob(os.path.join(RB_CROPS_PATH, '*/*'))

ALL_CROPS = INA_CROPS + RB_CROPS

# Build a fast lookup dictionary: {basename: full_path}
all_crops_dict = {os.path.basename(path).lower(): path for path in ALL_CROPS}

# Load data
df = pd.read_csv(CSV_FILE_PATH)

# Process data
annotations_subject_data = df[['annotations', 'subject_data']].copy()
annotations_subject_data['stage'] = annotations_subject_data['annotations'].apply(extract_value).str.lower()
annotations_subject_data['filename'] = annotations_subject_data['subject_data'].apply(extract_filename)
results = annotations_subject_data[['stage', 'filename']].dropna()

# Clean results: get most common stage per filename
cleaned_results = results.groupby('filename')['stage'].agg(lambda x: x.mode()[0]).reset_index()
stage_to_filenames_dict = cleaned_results.groupby('stage')['filename'].apply(list).to_dict()

# Copy images to class folders
for stage, filenames in stage_to_filenames_dict.items():
    output_path = os.path.join(DATASETS_PATH, 'cropped', 'zooniverse', 'labeled_images', 'classes', stage)

    os.makedirs(output_path, exist_ok=True)
    for image in tqdm(filenames, desc=f"Copying for stage {stage}"):
        src = all_crops_dict.get(image)
        dst = os.path.join(output_path, image)
        if src and os.path.exists(src):
            shutil.copy2(src, dst)
        else:
            print(f"Source file {src} does not exist. Skipping.")

### Manual task: control images in classes and create following structure

### The intended directory structure is the following:

```text
|-- classification/
    |-- train/
        |-- tagged/
            |-- prophase/
                |-- example_prophase_image.png
            |-- metaphase/
                |-- example_metaphase_image.png
            |-- anaphase/
                |-- example_anaphase_image.png
            |-- telophase/
                |-- example_telophase_image.png
        |-- untagged/
            |-- example_untagged_image.png
```

In [16]:
raise ValueError("Please organize images in the classes folders") #Intended to stop the script here to mannualy organize classes

ValueError: Please organize images in the classes folders

### Creation of train - test split

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

def split_dataset(source_dir, test_dir, test_size=0.2, seed=SEED):
    os.makedirs(test_dir, exist_ok=True)
    classes = [d for d in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, d))]
    for class_name in classes:
        class_path = os.path.join(source_dir, class_name)
        images = [f for f in os.listdir(class_path) if os.path.isfile(os.path.join(class_path, f))]
        train_imgs, test_imgs = train_test_split(images, test_size=test_size, random_state=seed)
        # Create class subfolder in test_dir
        test_class_dir = os.path.join(test_dir, class_name)
        os.makedirs(test_class_dir, exist_ok=True)
        # Copy test images
        for img in test_imgs:
            src = os.path.join(class_path, img)
            dst = os.path.join(test_class_dir, img)
            shutil.copy2(src, dst)
        print(f"Class '{class_name}': {len(test_imgs)} test images copied.")

TRAIN_PATH = os.path.join(VAE_DATASET_PATH,  'train', 'tagged')
TEST_PATH = os.path.join(VAE_DATASET_PATH,  'test')
split_dataset(TRAIN_PATH, TEST_PATH, test_size=0.2)


### Pending manual task

After the classes directories are created it is needed to augment the images. For that there is a notebook (data_augmentation.ipynb) which will have to be run for the classes and the untagged images three times for each. The prophase folder can be augmented only once since it is the class with the most amount of images

### Unused code

In [ ]:
import cv2
import os
from tqdm import tqdm

def resize_images_in_directory(dataset_path, output_dir, target_size=(200, 200)):
    """
    Searches for images starting with 'IMG' in a directory, resizes them,
    and saves them to an output directory.

    Args:
        dataset_path (str): The path to the directory containing the images.
        output_dir (str): The path to the directory where resized images will be saved.
        target_size (tuple): A tuple (width, height) for the resized images.
    """
    # Create the output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created directory: {output_dir}")

    # Check if the dataset path exists
    if not os.path.isdir(dataset_path):
        print(f"Error: The directory '{dataset_path}' was not found.")
        return

    print(f"Searching for images in '{dataset_path}'...")

    # Loop through all the files in the source directory
    for filename in tqdm(os.listdir(dataset_path)):
        # Check if the file name starts with 'IMG' and is a supported image format
        if filename.startswith('IMG') and filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
            input_filepath = os.path.join(dataset_path, filename)

            # Read the image using OpenCV
            image = cv2.imread(input_filepath)

            # Check if the image was loaded successfully
            if image is None:
                print(f"Warning: Could not read image {input_filepath}. Skipping.")
                continue

            # Resize the image
            # cv2.INTER_AREA is generally good for shrinking images.
            resized_image = cv2.resize(image, target_size, interpolation=cv2.INTER_AREA)

            # Construct the output file path
            output_filepath = os.path.join(output_dir, filename)

            # Save the resized image
            cv2.imwrite(output_filepath, resized_image)
            print(f"Resized and saved '{filename}' to '{output_filepath}'")

    print("\nProcessing complete.")

DATASET_PATH = os.path.join(CROPPED_PATH, 'd') # Run again for 'not' dataset
resize_images_in_directory(DATASET_PATH, DATASET_PATH)



In [ ]:

import os
import glob
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm
import cv2 as cv
import argparse


def images_to_parquet_sharded(image_dir, output_path, output_prefix="data_shard", shard_size_mb=100):
    """
    Converts a folder of images into multiple Parquet shard files (~shard_size_mb each).

    Args:
        image_dir (str): Path to dataset folder. If images are grouped in
                         subfolders, the folder name will be used as label.
        output_path (str): Path to processed dataset folder. 
        output_prefix (str): Prefix for output parquet files (e.g., "data_shard").
        shard_size_mb (int): Approximate shard size in MB.
    """
    shard_size_bytes = shard_size_mb * 1024 * 1024
    shard_index = 0
    records = []
    all_images_count = 0
    total_written = 0

    image_paths = glob.glob(os.path.join(image_dir, "**", "*.*"), recursive=True)

    for path in tqdm(image_paths, desc="Packing images"):
        ext = os.path.splitext(path)[1].lower()
        if ext not in [".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp"]:
            continue

        label = os.path.basename(os.path.dirname(path))

        img_bytes = cv.imread(path)
        img_bytes = cv.imencode('.png', img_bytes)[1].tobytes()

        records.append({"image": img_bytes, "label": label, "filename": os.path.basename(path)})
        total_written += len(img_bytes)

        # When current shard size exceeds limit, flush to disk
        if total_written >= shard_size_bytes:
            df = pd.DataFrame(records)
            table = pa.Table.from_pandas(df)
            out_path = os.path.join(output_path, f"{output_prefix}-{shard_index:05d}.parquet")
            pq.write_table(table, out_path)
            all_images_count += len(records)
            # Reset for next shard
            shard_index += 1
            records = []
            total_written = 0
            

    # Write leftover records
    if records:
        df = pd.DataFrame(records)
        table = pa.Table.from_pandas(df)
        out_path = os.path.join(output_path, f"{output_prefix}-{shard_index:05d}.parquet")
        pq.write_table(table, out_path)
        all_images_count += len(records)
        
    print(f"✅ Wrote {all_images_count} images to {output_path}")


In [ ]:
images_to_parquet_sharded('/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/controlnet_dataset/test/blurred_upscaled/', '/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/controlnet_dataset/test/blurred_upscaled/')


Packing images: 100%|██████████| 242/242 [00:00<00:00, 258.62it/s]

✅ Wrote 242 images to /home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/controlnet_dataset/test/blurred_upscaled/
